# Assignment Sesi 29 Tugas 2
Nama: Faraday Barr Fatahillah

**Tugas 2**

Anda bekerja di perusahaan Otomotif dan diberikan link [ini](https://autocatalogarchive.com/mitsubishi/) yang berisi informasi tentang beberapa mobil mitsubishi dengan berbagai macam bahasa dalam bentuk dokumen PDF. Dari katolog tersebut, gunakanlah semua file PDF yang berbahasa Indonesia atau berkode (ID), misalkan **2018 - Outlander Sport (ID)**.

Buatlah AI yang dapat yang dapat melakukan Product Search yang dapat menjawab atau mencari konteks yang cocok dengan input berikut:
- Detail spesifikasi Mitsubishi Destinator
- Mobil yang cocok untuk Travel dengan jumlah bangku atau *seating capacity* yang besar.
- Mobil untuk perjalanan jauh yang nyaman
- Mobil Mitsubishi yang irit bahan bakar
- Mobil Mitsubishi hybrid atau electric 

Lakukanlah pencarian dengan Hybrid Search dan gunakanlah Pinecone sebagai vector database.

In [ ]:
import pymupdf
import pdfplumber
import re
from pathlib import Path

In [ ]:
PDF_FOLDER = "/Session29_Tugas2_PDFS"

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Pull all text + tables from a PDF into one raw string.
    Uses PyMuPDF for text and pdfplumber for tables.
    """
    doc = pymupdf.open(pdf_path)
    pages_text = []
    for page in doc:
        text = page.get_text('text').strip()
        if text:
            pages_text.append(text)
    doc.close()

    tables_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in page.extract_tables():
                for row in table:
                    clean = [cell.strip() if cell else '' for cell in row]
                    if any(clean):
                        tables_text.append(' | '.join(clean))

    combined = '\n'.join(pages_text)
    if tables_text:
        combined += '\n\n--- TABLES ---\n' + '\n'.join(tables_text)

    return re.sub(r'\n{3,}', '\n\n', combined).strip()


pdf_files = sorted(Path(PDF_FOLDER).glob('*.pdf'))
print(f'Found {len(pdf_files)} PDF(s):')
for p in pdf_files:
    print(f'  • {p.name}')